# Writing a stage

Each step of CLPipe is a [ceci](https://github.com/LSSTDESC/ceci) stage, a
Python class that declares the files it reads and writes and its options, and
does its work in a `run` method. To add a step to the pipeline, you write a new
stage.

As an example, we write `CLPRichnessCut`, a stage that removes the lowest
richness bins from a SACC file, like the run without the first richness bin of
the CLPipe paper. It reads the data of the cosmoDC2 redMaPPer baseline analysis
in `data/clusters_sacc_file_cov.sacc`.

Use the `firecrown_clp` kernel, as explained in [README.md](README.md).

In [ ]:
import os
import shutil
import sys
sys.path.append("..")  # not needed if CLPipe is installed with `pip install -e .`

import numpy as np
import sacc
from ceci import PipelineStage
from ceci.config import StageParameter

from clpipe.file_types import SACCFile

## 1. The stage

A stage has four parts.
- `name` is the name of the stage in the pipeline and configuration files.
- `inputs` and `outputs` list the files, as pairs of a tag and a file type. The
  tags connect the stages of a pipeline. The file types are in
  `clpipe/file_types.py`.
- `config_options` lists the options, with their type, default and
  description. Their values are in `self.config`.
- `run` does the work. `self.get_input(tag)` gives the path of an input, and
  `self.get_output(tag, final_name=True)` the path where an output is written.

In [ ]:
# ceci records the file where each stage is defined. A notebook has no file,
# so we give it a name. This is not needed for a stage in a Python file.
sys.modules["__main__"].__file__ = "writing_a_stage.ipynb"


class CLPRichnessCut(PipelineStage):
    """Remove the data points of the richness bins below a given richness."""

    name = "CLPRichnessCut"
    allow_reload = True  # only needed in a notebook, to run this cell again

    inputs = [("clusters_sacc_file_cov", SACCFile)]
    outputs = [("clusters_sacc_file_cut", SACCFile)]
    config_options = {
        "min_richness": StageParameter(float, 20.0, msg="Richness bins starting below this value are removed."),
    }

    def run(self):
        data = sacc.Sacc.load_fits(self.get_input("clusters_sacc_file_cov"))

        # The richness bin edges are stored as log10(richness)
        low_bins = [
            name for name, tracer in data.tracers.items()
            if tracer.tracer_type == "bin_richness"
            and tracer.lower < np.log10(self.config["min_richness"])
        ]
        cut = [i for i, point in enumerate(data.data) if set(point.tracers) & set(low_bins)]

        # Removes the data points and the matching rows and columns of the covariance
        data.remove_indices(cut)
        data.save_fits(self.get_output("clusters_sacc_file_cut", final_name=True), overwrite=True)
        print(f"Removed {len(cut)} data points, in the richness bins {low_bins}")

## 2. Running it

`make_stage` builds the stage from Python values, the paths of its files and
its options. In a pipeline, ceci builds it from the pipeline and configuration
files, and calls `run` for you.

In [ ]:
os.makedirs("outputs", exist_ok=True)
stage = CLPRichnessCut.make_stage(
    clusters_sacc_file_cov="data/clusters_sacc_file_cov.sacc",
    clusters_sacc_file_cut="outputs/clusters_sacc_file_cut.sacc",
    min_richness=35.0,
)
stage.run()

In [ ]:
for path in ["data/clusters_sacc_file_cov.sacc", "outputs/clusters_sacc_file_cut.sacc"]:
    data = sacc.Sacc.load_fits(path)
    edges = sorted({tracer.lower for tracer in data.tracers.values() if tracer.tracer_type == "bin_richness"
                    and any(tracer.name in point.tracers for point in data.data)})
    print(f"{path}: {len(data.mean)} data points, richness bins starting at {[round(10**e) for e in edges]}")

## 3. Using it in a pipeline

In a pipeline file, the stage goes before `CLPFirecrown`. The `aliases` of
`CLPFirecrown` make it read the file written by `CLPRichnessCut` in place of its
usual `clusters_sacc_file_cov` input.
```yaml
stages:
  - name: CLPRichnessCut
    nprocess: 1
  - name: CLPFirecrown
    nprocess: 1
    aliases:
        clusters_sacc_file_cov: clusters_sacc_file_cut
```
The option of the stage goes in the configuration file, in a section with the
name of the stage.
```yaml
CLPRichnessCut:
    min_richness: 35.0
```

## 4. Adding it to CLPipe

1. Put the class in a new file of `clpipe/`, for instance
   `clpipe/clp_richness_cut.py`, without the two lines that are only needed in
   a notebook.
2. Import it in `clpipe/__init__.py`, so that ceci finds it in pipelines with
   `modules: clpipe`. It can then also run alone, from the command line
   ```bash
   python -m clpipe CLPRichnessCut --config=config.yml \
       --clusters_sacc_file_cov=clusters_sacc_file_cov.sacc \
       --clusters_sacc_file_cut=clusters_sacc_file_cut.sacc
   ```
3. Add tests in `tests/`. `tests/conftest.py` has mock SACC files, and
   `run_ceci_stage` runs a stage from the command line as in a pipeline. Mark
   the tests that need the DESC stack (CCL, Firecrown, CosmoSIS) with
   `@pytest.mark.slow`.
4. Add the module to `docs/source/api.rst`, so that its documentation is built
   from the docstrings.

Then open a pull request, as explained in the main
[README](https://github.com/LSSTDESC/CLPipe#contributing).